In [61]:
import re
import ast
import pandas as pd
import numpy as np

def _parse_kinase_coord(x):
    """
    Accepts things like:
      - [[13, 271]]
      - "[[13, 271]]"
      - "13,271"  (fallback)
    Returns list of (start,end) tuples (1-indexed, inclusive).
    """
    if isinstance(x, list):
        coord = x
    elif isinstance(x, str):
        s = x.strip()
        try:
            coord = ast.literal_eval(s)
        except Exception:
            nums = list(map(int, re.findall(r"-?\d+", s)))
            if len(nums) >= 2:
                coord = [[nums[0], nums[1]]]
            else:
                coord = []
    else:
        coord = []

    pairs = []
    for item in coord:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            pairs.append((int(item[0]), int(item[1])))
    return pairs

def _extract_15mer_centered(seq: str, p_ind_1: int, p_aa: str, pad_char: str = ""):
    """
    15-aa window centered at p_ind_1 (1-indexed). Inclusive window [p-7, p+7].
    Pads with pad_char if near termini.
    Returns (window, ok_match) where ok_match indicates seq[p_ind_1-1]==p_aa (if possible).
    """
    if not isinstance(seq, str) or len(seq) == 0 or pd.isna(p_ind_1):
        return (np.nan, False)

    p = int(p_ind_1)
    if p < 1:
        return (np.nan, False)

    ok = True
    if isinstance(p_aa, str) and len(p_aa) == 1 and p <= len(seq):
        ok = (seq[p-1] == p_aa)

    left = p - 7
    right = p + 7

    chars = []
    for i in range(left, right + 1):
        if 1 <= i <= len(seq):
            chars.append(seq[i-1])
        else:
            chars.append(pad_char)
    return ("".join(chars), ok)

def _extract_domain(seq: str, coords, join_char: str = ""):
    """
    Extract kinase domain based on coords = list of (start,end) 1-indexed inclusive.
    If multiple segments, concatenates them in order (default join_char="").
    """
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    if not coords:
        return np.nan

    parts = []
    for (start, end) in coords:
        if start < 1 or end < 1:
            continue
        if end < start:
            start, end = end, start
        parts.append(seq[start-1:end])
    if not parts:
        return np.nan
    return join_char.join(parts)

def _collapse_conflicting_labels(
    df_train: pd.DataFrame,
    key_cols=("kinase_seq", "substrate_seq"),
    label_col="label",
):
    """
    If the same (kinase_seq, substrate_seq) appears with both label 0 and 1,
    keep only the label==1 rows for that pair.

    - Does NOT merge/average scores; it just filters.
    - If a pair appears only with label 0 -> keep (all) 0 rows
      (typically there should be only one row, but we don't assume).
    - If a pair appears only with label 1 -> keep (all) 1 rows.
    """
    # Normalize label to numeric {0,1}
    df = df_train.copy()
    df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
    df = df.dropna(subset=[label_col])
    df[label_col] = df[label_col].astype(int)

    # Identify keys that have any positive
    has_pos = (
        df.groupby(list(key_cols))[label_col]
          .max()
          .rename("_has_pos")
          .reset_index()
    )
    df = df.merge(has_pos, on=list(key_cols), how="left")

    # If a key has a positive, drop the negatives for that key
    df = df.loc[~((df["_has_pos"] == 1) & (df[label_col] == 0))].copy()
    df = df.drop(columns=["_has_pos"], errors="ignore").reset_index(drop=True)
    return df

def make_training_sequences(
    df_raw: pd.DataFrame,
    substrate_seq_col: str = "substrate_seq",
    kinase_seq_col: str = "kinase_seq",
    p_ind_col: str = "p_ind",
    p_aa_col: str = "p_aa",
    kinase_coord_col: str = "kinase_coord",
    label_col: str = "label",
    pad_char: str = "",
    drop_mismatch: bool = False,
    collapse_conflicting_labels: bool = True,
):
    """
    Returns:
      df_out with new columns:
        - substrate_seq_15aa
        - kinase_domain_seq
        - p_aa_match (bool)
      plus a minimal df_train with columns:
        - kinase_seq (domain)
        - substrate_seq (15aa)
        - label

    Extra behavior:
      - If collapse_conflicting_labels=True, then for any (kinase_seq, substrate_seq)
        pair that has both label 0 and 1, only keep the label==1 rows.
    """
    df = df_raw.copy()

    # parse kinase coords
    df["_kinase_pairs"] = df[kinase_coord_col].apply(_parse_kinase_coord)

    # kinase domain seq
    df["kinase_domain_seq"] = [
        _extract_domain(seq, pairs)
        for seq, pairs in zip(df[kinase_seq_col], df["_kinase_pairs"])
    ]

    # substrate 15mer
    out = [
        _extract_15mer_centered(seq, pind, paa, pad_char=pad_char)
        for seq, pind, paa in zip(df[substrate_seq_col], df[p_ind_col], df[p_aa_col])
    ]
    df["substrate_seq_15aa"] = [x[0] for x in out]
    df["p_aa_match"] = [x[1] for x in out]

    # optionally drop rows where p_aa doesn't match the sequence at p_ind
    if drop_mismatch:
        df = df[df["p_aa_match"] == True].copy()

    # minimal training df for your pipeline
    df_train = df[["kinase_domain_seq", "substrate_seq_15aa", label_col]].copy()
    df_train = df_train.rename(
        columns={
            "kinase_domain_seq": "kinase_seq",
            "substrate_seq_15aa": "substrate_seq",
            label_col: "label",
        }
    )

    # drop NAs created by bad coords / bad indices
    df_train = df_train.dropna(subset=["kinase_seq", "substrate_seq", "label"]).reset_index(drop=True)

    # NEW: remove conflicts (keep only label==1 if both 0 and 1 exist for same pair)
    if collapse_conflicting_labels:
        df_train = _collapse_conflicting_labels(
            df_train,
            key_cols=("kinase_seq", "substrate_seq"),
            label_col="label",
        )

    # cleanup helper
    df = df.drop(columns=["_kinase_pairs"], errors="ignore")

    return df, df_train


### Prepare training data v1: DB only

In [71]:
# dat = pd.read_pickle('../kinase_ml/data/training_data/interations_e2e_1n1p_db_classed_random_lysate_random_withacc_230621_train.pkl').groupby('label').head(2000)
dat = pd.read_pickle('./training_data/training_data/interations_e2e_1n1p_classed_random_withacc_230621.pkl')

In [73]:
# df_raw = your dataframe with the columns you listed
df_aug, df_train = make_training_sequences(dat, drop_mismatch=False)
# df_train now has exactly: kinase_seq (domain), substrate_seq (15aa), label
print(df_train.head())

                                          kinase_seq    substrate_seq  label
0  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  MKKKDEGSYDLGKKP      1
1  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  FPLRKTASEPNLKVR      1
2  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  PLLARSPSTNRKYPP      1
3  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  QAGSHSNSFRLSNGR      1
4  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  GRGGRGGSRARNLPL      1


In [74]:
df_train.to_csv('./pairs_v1.csv')
df_aug.to_csv('./pairs_aug_v1.csv')

In [14]:
df_train.shape

(119468, 3)

In [1]:
# python fold_kinases_from_pair_csv.py \
#   --input pairs_v1.csv \
#   --out_dir esmfold_out \
#   --batch_size 1 \
#   --skip_existing

In [2]:
# python prepare_splits.py \
#   --input_csv pairs_v1.csv \
#   --out_dir splits_v1 \
#   --seed 0 --val_frac 0.1 --test_frac 0.1 \
#   --dedupe_pairs

### Prepare training data v2: DB + Lyaste

In [78]:
dat = pd.read_pickle('./training_data/training_data/interations_e2e_1n1p_db_classed_random_lysate_random_withacc_230621.pkl')
# df_raw = your dataframe with the columns you listed
df_aug, df_train = make_training_sequences(dat, drop_mismatch=False)
# df_train now has exactly: kinase_seq (domain), substrate_seq (15aa), label
print(df_train.head())

                                          kinase_seq    substrate_seq  label
0  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  MKKKDEGSYDLGKKP      1
1  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  FPLRKTASEPNLKVR      1
2  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  PLLARSPSTNRKYPP      1
3  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  QAGSHSNSFRLSNGR      1
4  FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...  GRGGRGGSRARNLPL      1


In [79]:
df_train.to_csv('./pairs_v2.csv')
df_aug.to_csv('./pairs_aug_v2.csv')

In [ ]:
# python make_splits_v2_no_leak.py \
#   --expanded_csv pairs_v2.csv \
#   --splits_v1_dir splits_v1 \
#   --splits_v2_dir splits_v2 \
#   --key_cols kinase_seq,substrate_seq \
#   --dedup_before_split \
#   --seed 42

In [1]:
# python fold_kinases_from_pair_csv.py \
#   --input pairs_v2.csv \
#   --out_dir esmfold_out \
#   --batch_size 1 \
#   --skip_existing

### Prepare for training on human and testing on non-human 

In [28]:
import pandas as pd
all_kinase_organism_info = pd.read_csv('../data/interactions/interactions/all_kinase_organism_info_old.tsv',sep = '\t').iloc[:,[0,1,3,4,5]]

In [31]:
missing_kinase_organism_info = pd.read_csv('../data/interactions/interactions/idmapping_2025_12_22.tsv',sep = '\t').iloc[:,[0,1,2,3,5]]

In [33]:
all_kinase_organism_info = pd.concat([all_kinase_organism_info,missing_kinase_organism_info])

In [34]:
df_aug = pd.read_csv('../data/pairs_aug_v2.csv').iloc[:,1:]

In [37]:
df_aug['kinase'].isin(all_kinase_organism_info['From']).all()

np.True_

In [41]:
df_aug_w_organism = pd.merge(df_aug,all_kinase_organism_info,left_on = 'kinase',right_on = 'From')

In [43]:
df_aug_w_organism['is_human'] = df_aug_w_organism['Organism']=='Homo sapiens (Human)'

In [46]:
df_aug_w_organism.groupby(['is_human','label']).count().iloc[:,1]

is_human  label
False     0         45462
          1         45462
True      0        217842
          1        217842
Name: substrate_seq, dtype: int64

In [47]:
mkdir ../data/splits_human_nonhuman

In [67]:
df_aug_w_human = df_aug_w_organism.loc[df_aug_w_organism['is_human'],:]
df_aug_n_human = df_aug_w_organism.loc[~df_aug_w_organism['is_human'],:]

In [69]:
df_aug_human, df_train_human = make_training_sequences(df_aug_w_human, drop_mismatch=True)
df_train_human.to_csv('../data/pairs_v2_human.csv')
df_aug_human.to_csv('../data/pairs_aug_v2_human.csv')

In [73]:
df_aug_nhuman, df_train_nhuman = make_training_sequences(df_aug_n_human, drop_mismatch=True)
df_train_nhuman.to_csv('../data/pairs_v2_nonhuman.csv')
df_aug_nhuman.to_csv('../data/pairs_aug_v2_nonhuman.csv')

In [ ]:
# python prepare_splits.py \
#   --input_csv ../../data/pairs_v2_human.csv \
#   --out_dir ../../data/splits_human_nonhuman/ \
#   --seed 0 --val_frac 0.1 --test_frac 0.0 \
#   --dedupe_pairs

# python prepare_splits.py \
#   --input_csv ../../data/pairs_v2_nonhuman.csv \
#   --out_dir ../../data/tmp/ \
#   --seed 0 --val_frac 0.0 --test_frac 0.0 \
#   --dedupe_pairs

# rm ../../data/tmp/val.csv
# rm ../../data/tmp/test.csv
# mv ../../data/tmp/train.csv ../../data/tmp/test.csv         
# mv ../../data/tmp/test.csv ../../data/splits_human_nonhuman/
# rm -r ../../data/tmp

### Prepare for training on non-dark and testing on dark kinases

In [75]:
dark_kinome = pd.read_csv("../data/interactions/interactions/dark_kinases.csv",header = None)
dark_kinome.columns = ["Entry"]
dat = pd.read_csv('../data/pairs_aug_v2.csv').iloc[:,1:]

In [76]:
nodark = dat.loc[~(dat["kinase"].isin(dark_kinome["Entry"])),:]
dark = dat.loc[(dat["kinase"].isin(dark_kinome["Entry"])),:]

In [77]:
dark.shape

(39572, 11)

In [78]:
nodark.shape

(487036, 11)

In [79]:
df_aug_nodark, df_train_nodark = make_training_sequences(nodark, drop_mismatch=True)
df_train_nodark.to_csv('../data/pairs_v2_nodark.csv')
df_aug_nodark.to_csv('../data/pairs_aug_v2_nodark.csv')

In [80]:
df_aug_dark, df_train_dark = make_training_sequences(dark, drop_mismatch=True)
df_train_dark.to_csv('../data/pairs_v2_dark.csv')
df_aug_dark.to_csv('../data/pairs_aug_v2_dark.csv')

In [82]:
df_train_dark.groupby('label').count()

,kinase_seq,substrate_seq
label,,
0,19783,19783
1,19786,19786


In [3]:
# python prepare_splits.py \
#   --input_csv ../../data/pairs_v2_nodark.csv \
#   --out_dir ../../data/splits_dark_nondark/ \
#   --seed 0 --val_frac 0.1 --test_frac 0.0 \
#   --dedupe_pairs

# python prepare_splits.py \
#   --input_csv ../../data/pairs_v2_dark.csv \
#   --out_dir ../../data/tmp/ \
#   --seed 0 --val_frac 0.0 --test_frac 0.0 \
#   --dedupe_pairs

# rm ../../data/tmp/val.csv
# rm ../../data/tmp/test.csv
# mv ../../data/tmp/train.csv ../../data/tmp/test.csv         
# mv ../../data/tmp/test.csv ../../data/splits_dark_nondark/
# rm -r ../../data/tmp

In [1]:
import torch

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Total CUDA Devices: {num_gpus}")
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"Device {i}: {props.name} (Memory: {props.total_memory / 1024**2:.0f}MB)")
else:
    print("No CUDA devices found.")

Total CUDA Devices: 1
Device 0: NVIDIA A100-SXM4-40GB (Memory: 40442MB)
